In [1]:
import requests
import firebase_admin
from firebase_admin import credentials, db
from geopy.distance import distance as geopy_distance
import time
import requests

from geopy.distance import geodesic
from geopy.distance import great_circle

In [2]:
cred = credentials.Certificate("config_blindglasses.json")
firebase_admin.initialize_app(cred, {
    'databaseURL': 'https://blindglasses-cf714-default-rtdb.firebaseio.com/'
})

In [3]:
ref = db.reference('users')
data = ref.get()
print(data)

{'example-user-123': {'instructions': {'current_instruction': '666 meters to Turn left onto Swami Vivekanand Marg', 'step': 2, 'total_steps': 4}, 'location': {'current': {'lat': 18.4640777, 'lng': 73.8677695}, 'end': {'lat': 18.468861, 'lng': 73.8644874}, 'start': {'lat': 18.5204, 'lng': 73.8567}}}}


In [4]:
end_data = db.reference('users/example-user-123/location/end').get()
end_lat = end_data['lat']
end_lng = end_data['lng']

In [5]:
def fetch_route(current_lat, current_lng):
    url = f"https://router.project-osrm.org/route/v1/walking/{current_lng},{current_lat};{end_lng},{end_lat}?overview=full&geometries=geojson&steps=true"
    
    response = requests.get(url)
    data = response.json()

    if 'routes' not in data or not data['routes']:
        print("❌ No route found. Full response:")
        print(data)  # This helps debug if the coordinates are wrong
        return []

    return data['routes'][0]['legs'][0]['steps']

def get_current_location():
    current_data = db.reference('users/example-user-123/location/current').get()
    return current_data['lat'], current_data['lng']

In [6]:
steps = fetch_route(*get_current_location())
step_index = 0

In [ ]:
from geopy.distance import geodesic as geopy_distance
import time

step_index = 0

# Save total steps in Firebase before starting
db.reference('users/example-user-123/instructions').update({'total_steps': len(steps)})

while step_index < len(steps):
    current_lat, current_lng = get_current_location()
    step = steps[step_index]
    maneuver_lat, maneuver_lng = step['maneuver']['location'][1], step['maneuver']['location'][0]

    dist = geopy_distance((current_lat, current_lng), (maneuver_lat, maneuver_lng)).meters
    dist = round(dist)

    maneuver = step['maneuver']
    road_name = step['name']
    maneuver_type = maneuver.get('type', '').lower()
    modifier = maneuver.get('modifier', '').lower()

    # Build readable instruction
    if maneuver_type == "depart":
        base_instruction = "Depart"
    elif maneuver_type == "arrive":
        base_instruction = "You have arrived"
    elif maneuver_type == "turn":
        base_instruction = f"Turn {modifier}"
    elif maneuver_type == "continue":
        base_instruction = "Continue straight"
    elif maneuver_type == "roundabout":
        base_instruction = "Enter roundabout"
    else:
        base_instruction = maneuver_type.capitalize()

    # Add road name if available
    if road_name:
        base_instruction += f" onto {road_name}"

    full_instruction = f"{dist} meters to {base_instruction}"

    print(f"🚶 Step {step_index+1}/{len(steps)} → {full_instruction}")

    # Update Firebase with current distance-based instruction
    db.reference('users/example-user-123/instructions').update({
        'current_instruction': full_instruction,
        'step': step_index + 1
    })

    if dist < 7:
        print(f"📢 Now: {base_instruction}")
        step_index += 1

    time.sleep(2)


🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 119 meters to Depart
🚶 Step 1/6 → 44 meters to Depart
🚶 Step 1/6 → 44 meters to Depart
🚶 Step 1/6 → 41 meters to Depart
🚶 Step 1/6 → 41 meters to Depart
🚶 Step 1/6 → 42 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/6 → 42 meters to Depart
🚶 Step 1/6 → 42 meters to Depart
🚶 Step 1/6 → 43 meters to Depart
🚶 Step 1/